# Bifrost Getting Started — Groq + Mistral + Jina + Qdrant Cloud

### A free-tier, self-hosted AI gateway walkthrough

---

**Purpose:** Get hands-on with [Bifrost](https://github.com/maximhq/bifrost) (self-hosted, open-source AI
gateway by Maxim AI) using only providers with usable free tiers:

- **Groq** — chat/completions (LPU inference, very fast, generous free tier)
- **Mistral** — chat/completions (La Plateforme free tier)
- **Jina AI** — embeddings (used only in the RAG section, called directly — not via Bifrost)
- **Qdrant Cloud** — vector store (managed, used only in the RAG section)

**Your setup:** a single, already-created Docker container named `bifrost`, started with
`docker start bifrost` and nothing else — no `docker-compose`, no mounted `config.json`, no
extra Jaeger/Qdrant containers. Everything Bifrost-side is configured through its **Web UI**
at `http://localhost:8080`, which persists inside that same container across `docker stop` /
`docker start` cycles (as long as you don't delete and recreate the container).

**What you'll cover:**
1. Baseline: vanilla Groq/Mistral SDKs (no gateway)
2. Bifrost setup via the Web UI & a drop-in client
3. Bifrost feature showcase: unified API, failover, load balancing, streaming, observability, virtual keys, MCP
4. A small RAG pipeline built by hand (Jina embeddings + Qdrant Cloud + Bifrost generation)

**Prerequisites:**
1. Python 3.10+ and packages from `requirements.txt` (`pip install -r requirements.txt`)
2. `.env` filled in — `GROQ_API_KEY`, `MISTRAL_API_KEY` required from Section 1 onward; `JINA_API_KEY`,
   `QDRANT_URL` (your Qdrant Cloud cluster URL) and `QDRANT_API_KEY` required only for Section 4
3. `docker start bifrost` — that's it, nothing else to run
4. Groq + Mistral added as providers in the Bifrost Web UI (Section 2.2 walks through this)

> **Note:** This notebook skips Bifrost's *built-in* semantic caching (which needs an embedding
> model wired into Bifrost itself, and a vector store registered with Bifrost). Instead, Section 4
> builds a small RAG pipeline by hand with Jina + Qdrant Cloud so every step is visible, and Bifrost
> is only used for the final generation call.

---


---
<a id="section-0"></a>
## Section 0: Environment Setup & Prerequisites


### 0.1 Install Dependencies

Installs everything this notebook needs — Groq/Mistral SDKs, the OpenAI SDK (used as Bifrost's
drop-in client), Qdrant client, and `python-dotenv`. Run once per environment.


In [ ]:
%%capture
!pip install -r requirements.txt --quiet


### 0.2 Load Environment Variables

Reads your keys from `.env` so nothing is hardcoded in the notebook. `GROQ_API_KEY` and
`MISTRAL_API_KEY` are asserted here because everything from Section 1 onward needs them —
better to fail fast with a clear message than hit a confusing 401 five cells later.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY         = os.getenv("GROQ_API_KEY")
MISTRAL_API_KEY      = os.getenv("MISTRAL_API_KEY")
BIFROST_BASE_URL     = os.getenv("BIFROST_BASE_URL", "http://localhost:8080")
BIFROST_VIRTUAL_KEY  = os.getenv("BIFROST_VIRTUAL_KEY", "")

assert GROQ_API_KEY,    "Missing GROQ_API_KEY — set it in .env"
assert MISTRAL_API_KEY, "Missing MISTRAL_API_KEY — set it in .env"

print("Groq and Mistral keys loaded. Bifrost base URL:", BIFROST_BASE_URL)


### 0.3 Bifrost Health Check

Use case: a quick sanity check that the `bifrost` container is actually up and reachable on
port 8080 before anything else in this notebook tries to call it — saves you debugging a wall
of gateway errors that are really just "the container isn't running".


In [ ]:
import httpx

def check_bifrost_health():
    try:
        r = httpx.get(f"{BIFROST_BASE_URL}/health", timeout=3)
        return r.status_code == 200
    except Exception:
        return False

bifrost_ok = check_bifrost_health()
print(f"Bifrost: {'Online' if bifrost_ok else 'Offline'}")

if not bifrost_ok:
    print("WARNING: Bifrost is offline. Section 3 demos below will fail.")
    print("Start it with: docker start bifrost")


### 0.4 Shared Test Prompts & Models

Defines one set of prompts and model names reused across the whole notebook, so the vanilla-SDK
baseline, the Bifrost calls, and the RAG section are all comparing apples to apples.


In [ ]:
TEST_PROMPTS = {
    "simple":     "What is the capital of France?",
    "reasoning":  "Explain the difference between RAG and fine-tuning in 3 bullet points.",
    "code":       "Write a Python function that validates an email address using regex.",
    "duplicate1": "What is LangChain used for?",
    "duplicate2": "What is LangChain primarily used for?",
    "deepwiki":   "What are the stream modes in the new langgraph version? Use the deepwiki tool to check the langchain-ai/langgraph repo.",
    "tavily":     "Search the web for the latest news about Groq AI and summarize the top result."
}

# Free-tier models. Groq is retiring llama-3.3-70b-versatile and llama-3.1-8b-instant on
# 2026-08-16 (free/developer tier) in favor of these — using the replacements now so this
# notebook doesn't break in a month. Mistral's -latest aliases need no such change.
GROQ_MODEL          = "openai/gpt-oss-120b"   # capable — Groq's recommended replacement for llama-3.3-70b-versatile
GROQ_MODEL_FALLBACK = "openai/gpt-oss-20b"    # fast/cheap — replacement for llama-3.1-8b-instant
MISTRAL_MODEL          = "mistral-large-latest"
MISTRAL_MODEL_FALLBACK = "mistral-small-latest"

# Note: "openai/gpt-oss-120b" is OpenAI's *open-weight* model served on Groq's own hardware —
# it runs on GROQ_API_KEY, not OPENAI_API_KEY. Nothing in this notebook uses your OpenAI key
# or credits; Groq is the primary provider throughout, Mistral is the secondary/fallback.

import time
import json as _json

print(f"Groq model:      {GROQ_MODEL}  (fallback: {GROQ_MODEL_FALLBACK})")
print(f"Mistral model:   {MISTRAL_MODEL}  (fallback: {MISTRAL_MODEL_FALLBACK})")


---
<a id="section-1"></a>
## Section 1: Baseline — Vanilla Groq & Mistral SDKs

> **Goal:** Establish the raw baseline. Show what you have to write yourself WITHOUT a gateway —
> two separate SDKs, two separate error-handling paths, no shared observability.


### 1.1 Vanilla Groq SDK — Basic Chat Completion

Calls Groq directly with its own SDK, no gateway involved. This is the reference point every
later section (Bifrost, RAG) gets compared against — same prompt, same shape of result.


In [ ]:
from groq import Groq

client_groq = Groq(api_key=GROQ_API_KEY)

def vanilla_groq_call(prompt: str) -> dict:
    start = time.perf_counter()
    response = client_groq.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    latency_ms = (time.perf_counter() - start) * 1000
    return {
        "content":    response.choices[0].message.content,
        "latency_ms": round(latency_ms, 2),
        "tokens":     response.usage.total_tokens
    }

result = vanilla_groq_call(TEST_PROMPTS["simple"])
print(result)


### 1.2 Vanilla Mistral SDK — Basic Chat Completion

Same idea, on Mistral's own SDK. Notice the different client shape (`chat.complete` vs
`chat.completions.create`) — this per-provider inconsistency is exactly what a gateway hides.


In [ ]:
# Note: in current mistralai SDK versions (2.x), Mistral moved to mistralai.client —
# `from mistralai import Mistral` now raises ImportError. Use this import instead.
from mistralai.client import Mistral

client_mistral = Mistral(api_key=MISTRAL_API_KEY)

def vanilla_mistral_call(prompt: str) -> dict:
    start = time.perf_counter()
    response = client_mistral.chat.complete(
        model=MISTRAL_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    latency_ms = (time.perf_counter() - start) * 1000
    return {
        "content":    response.choices[0].message.content,
        "latency_ms": round(latency_ms, 2),
        "tokens":     response.usage.total_tokens
    }

result = vanilla_mistral_call(TEST_PROMPTS["reasoning"])
print(result)


### 1.3 Vanilla Streaming — Groq

Use case: chat UIs that need to render tokens as they arrive instead of waiting for the full
response. Shown here on the raw SDK so Section 3.4 can show the same thing through Bifrost.


In [ ]:
def vanilla_groq_stream(prompt: str):
    stream = client_groq.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            print(delta, end="", flush=True)
    print()

vanilla_groq_stream(TEST_PROMPTS["code"])


### 1.4 What's Missing Without a Gateway

| Problem                   | Vanilla Groq SDK | Vanilla Mistral SDK |
|---------------------------|-------------------|-----------------------|
| Automatic failover        | Manual            | Manual                |
| Multi-provider routing    | Not possible      | Not possible           |
| Unified request/response  | Per-SDK           | Per-SDK                |
| Rate limit handling       | Manual retry      | Manual retry           |
| Observability / tracing   | External only     | External only          |
| Budget enforcement        | Not possible      | Not possible            |
| MCP tool governance        | Not possible      | Not possible            |
| Streaming normalization   | Per-provider       | Per-provider            |


### 1.5 Manual Fallback — What You'd Have to Write Yourself

This is the fallback logic teams end up hand-writing without a gateway: try Groq, catch the
failure, retry on Mistral. Section 3.2 replaces this whole function with a single HTTP header.


In [ ]:
def naive_manual_fallback(prompt: str) -> str:
    try:
        response = client_groq.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    except Exception as primary_error:
        print(f"Primary provider (Groq) failed: {primary_error}")
        try:
            response = client_mistral.chat.complete(
                model=MISTRAL_MODEL,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content
        except Exception as fallback_error:
            raise RuntimeError(f"All providers failed: {fallback_error}")

print("=== Manual Fallback Demo (what a gateway replaces) ===")
result = naive_manual_fallback(TEST_PROMPTS["simple"])
print(f"Result: {result[:200]}")


---
<a id="section-2"></a>
## Section 2: Bifrost Setup & Architecture

> **Bifrost** is a Go-based, open-source AI gateway. It provides failover, load balancing,
> virtual keys, MCP gateway, and observability — all via a single self-hosted container.


### 2.1 Your Bifrost Container

You already have a container named `bifrost`. The full lifecycle for this notebook is just:

```bash
docker start bifrost   # bring it up
docker stop bifrost    # shut it down when you're done
```

No `docker run`, no `-v config.json:...`, no `docker-compose`, no separate Qdrant/Jaeger
containers — none of that is needed for Sections 0–3. Bifrost persists whatever you configure
through its Web UI **inside that same container**, so it survives `stop`/`start` cycles fine.
It would only be lost if the container itself were deleted and recreated (`docker rm` + a fresh
`docker run`) without a mounted data volume — not something this notebook asks you to do.

**Do you need a Bifrost API key?** No. Bifrost does not require its own gateway-level
credential for incoming requests by default — that's a separate, optional layer (Virtual Keys,
Section 3.7) you can turn on later for budget/access control. The OpenAI SDK below still wants
*some* string in `api_key` (it's a required constructor argument), so we pass a placeholder
unless you've created a virtual key.


### 2.2 Add Groq & Mistral as Providers (Web UI)

Since there's no `config.json` mounted, providers are added through the Web UI instead of a
file — and because your container only ever gets `docker start`ed (never recreated), this
sticks around permanently:

1. Open `http://localhost:8080`
2. Go to **Providers** → **Add Provider**
3. Provider = `groq`, paste your actual `GROQ_API_KEY` value directly into the key field
   (not `env.GROQ_API_KEY` — that syntax only resolves an env var that was set when the
   container was *created*, which doesn't apply to your `docker start`-only workflow)
4. Repeat for provider = `mistral` with your `MISTRAL_API_KEY`
5. (Optional, for Section 3.8) Go to **MCP Gateway → MCP Catalog** → **Add MCP Server**:
   - `deepwiki` — connection type `HTTP (Streamable)`, connection URL `https://mcp.deepwiki.com/mcp`, auth `None`
   - `tavily` — same connection type, connection URL `https://mcp.tavily.com/mcp/?tavilyApiKey=YOUR_TAVILY_KEY`
     (free key from `https://app.tavily.com/home`), auth `None`
   - **Important:** after creating each, open it and toggle **Auto-execute** on for its tools.
     Without this, Bifrost proposes the tool call and stops — it won't actually run the tool or
     return a final answer.


### 2.3 Bifrost Drop-in Client Setup

One `OpenAI` client instance now talks to every provider Bifrost knows about — which provider
handles a given call is picked purely by the `"<provider>/<model>"` string, not by which client
you instantiate. This is the core "drop-in" pitch: point any OpenAI-compatible client at
Bifrost's `/v1` endpoint and keep writing normal OpenAI SDK code.


In [ ]:
from openai import OpenAI

bifrost = OpenAI(
    api_key=BIFROST_VIRTUAL_KEY or "not-needed-unless-virtual-keys-enabled",
    base_url=f"{BIFROST_BASE_URL}/v1"
)

print("Bifrost client initialized — provider is chosen per-call via the model string, e.g.:")
print(f'  "groq/{GROQ_MODEL}"')
print(f'  "mistral/{MISTRAL_MODEL}"')


---
<a id="section-3"></a>
## Section 3: Bifrost Feature Showcase


### 3.1 Feature 1 — Unified Drop-in API (Groq + Mistral, Same Client)

Use case: swap providers per-request — e.g. cheap/fast Groq for simple queries, Mistral for
others — without maintaining two SDKs or two code paths in your application.


In [ ]:
def bifrost_call(prompt: str, model: str) -> dict:
    start = time.perf_counter()
    raw = bifrost.chat.completions.with_raw_response.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    latency_ms = (time.perf_counter() - start) * 1000
    response = raw.parse()
    extra = _json.loads(raw.text).get("extra_fields", {})
    return {
        "content":        response.choices[0].message.content,
        "provider":       extra.get("provider", "unknown"),
        "latency_ms":     round(latency_ms, 2),
        "tokens":         response.usage.total_tokens
    }

print("=== Groq via Bifrost ===")
print(bifrost_call(TEST_PROMPTS["simple"], f"groq/{GROQ_MODEL}"))

print()
print("=== Mistral via Bifrost (same client, just a different model string) ===")
print(bifrost_call(TEST_PROMPTS["simple"], f"mistral/{MISTRAL_MODEL}"))


### 3.2 Feature 2 — Automatic Failover / Provider Fallback

Use case: keep serving requests even if your primary provider is down or rate-limiting you.
Bifrost retries on the fallback model automatically — your application code never sees the
failure. It triggers on **5xx errors, 429 rate limits, and 401 auth failures**; it does **not**
fall back on **404 (model not found)** or a missing provider config.

> That second bullet is easy to get backwards, so it's tested explicitly below rather than
> asserted: a nonexistent model name comes back as a 404 pass-through, *not* a fallback —
> confirmed against this exact Bifrost instance while writing this notebook. Reliably forcing a
> real 401/429/5xx would mean temporarily breaking your live Groq key in the Web UI, which isn't
> worth doing just for a demo — the 404 case already proves the boundary.


In [ ]:
def bifrost_failover_demo(prompt: str, primary_model: str, fallback_model: str) -> dict:
    response = httpx.post(
        f"{BIFROST_BASE_URL}/v1/chat/completions",
        headers={
            "Content-Type": "application/json",
            "x-bifrost-fallback-models": fallback_model
        },
        json={
            "model":    primary_model,
            "messages": [{"role": "user", "content": prompt}]
        },
        timeout=30
    )
    data = response.json()
    extra = data.get("extra_fields", {})
    provider = extra.get("provider", "unknown")
    return {
        "content":       data.get("choices", [{}])[0].get("message", {}).get("content", str(data)),
        "provider_used": provider,
        "model_used":    extra.get("resolved_model_used", "unknown"),
        "was_fallback":  bool(provider) and provider != "unknown" and not primary_model.startswith(provider)
    }

print("=== Normal call (Groq primary, Mistral fallback configured) ===")
result = bifrost_failover_demo(TEST_PROMPTS["simple"], f"groq/{GROQ_MODEL}", f"mistral/{MISTRAL_MODEL}")
print(result)

print()
print("=== Bad model name (404) — does NOT trigger the Mistral fallback, by design ===")
result = bifrost_failover_demo(TEST_PROMPTS["simple"], "groq/not-a-real-model", f"mistral/{MISTRAL_MODEL}")
print(result)


### 3.3 Feature 3 — Weighted Load Balancing Across API Keys

Use case: spread traffic across multiple keys for the *same* provider to raise your effective
rate limit — e.g. two free-tier Groq keys instead of one. Add a second key under the `groq`
provider in the Web UI with its own weight to see traffic split; with a single key configured
(the default here) every request just uses that one key, which this cell confirms.


In [ ]:
def bifrost_load_balance_info(model: str):
    raw = bifrost.chat.completions.with_raw_response.create(
        model=model,
        messages=[{"role": "user", "content": "Hello"}]
    )
    raw.parse()
    extra = _json.loads(raw.text).get("extra_fields", {})
    print(f"Provider       : {extra.get('provider', 'unknown')}")
    print(f"Model used     : {extra.get('resolved_model_used', 'unknown')}")
    print(f"Model requested: {extra.get('original_model_requested', 'unknown')}")
    print(f"Latency (ms)   : {extra.get('latency', 'unknown')}")

print("=== Weighted Load Balancing (Bifrost) ===")
print("Add a second key under the groq provider in the Web UI (with a weight) to split traffic")
bifrost_load_balance_info(f"groq/{GROQ_MODEL}")


### 3.4 Feature 4 — Streaming (Unified Across Providers)

Use case: same streaming loop, no `if provider == "groq"` branching — swap the model string
and the same code streams from a different backend.


In [ ]:
def bifrost_stream(prompt: str, model: str):
    stream = bifrost.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)
    print()

print("=== Groq via Bifrost (streaming) ===")
bifrost_stream(TEST_PROMPTS["code"], f"groq/{GROQ_MODEL}")

print("\n=== Mistral via Bifrost (same client code!) ===")
bifrost_stream(TEST_PROMPTS["code"], f"mistral/{MISTRAL_MODEL}")


### 3.5 Feature 5 — Observability: Logs API

Use case: debugging — see exactly what Bifrost routed, to which provider, and how long it took,
without adding any logging of your own. Full Prometheus `/metrics` needs a separate plugin
binary (not covered here); this REST endpoint works out of the box.


In [ ]:
def bifrost_get_logs():
    try:
        response = httpx.get(f"{BIFROST_BASE_URL}/api/logs?limit=5", timeout=5)
        if response.status_code == 200:
            return response.json()
        return {"note": "logs endpoint unavailable", "status": response.status_code}
    except Exception as e:
        return {"error": str(e)}

import pprint
pprint.pprint(bifrost_get_logs())


### 3.6 Feature 6 — OpenTelemetry Tracing (Optional — Skip if You Don't Want Extra Infra)

Use case: distributed tracing when Bifrost is one hop in a bigger system. This needs an OTEL
collector (e.g. Jaeger) running and Bifrost's `otel` plugin pointed at it via the Web UI's
Plugins tab — infra you explicitly don't want to stand up right now. The call below still
succeeds either way (Bifrost just has nowhere to export the trace to), so it's safe to run,
but there won't be anywhere to view it. Feel free to skip this cell entirely.


In [ ]:
raw = bifrost.chat.completions.with_raw_response.create(
    model=f"groq/{GROQ_MODEL}",
    messages=[{"role": "user", "content": "Say hello and confirm you received this message."}],
    extra_headers={"x-bifrost-trace-id": "notebook-demo-001"}
)
response = raw.parse()
print("Trace ID injected: notebook-demo-001")
print("(No trace viewer running — this header is a no-op unless the otel plugin + a collector are configured)")
print(f"Response: {response.choices[0].message.content[:100]}...")


### 3.7 Feature 7 — Virtual Keys & Budget Governance

Use case: hand a teammate, or a specific app, a scoped key with a spending cap — instead of
giving out your raw Groq/Mistral keys. Create one at `http://localhost:8080` → **Virtual Keys**
(this is a separate tab from **Providers** — a name you gave a *provider* key, e.g. when adding
Groq/Mistral, is not a virtual key and won't be enforced as one). Once created, Bifrost shows
you the generated secret once — put that in `BIFROST_VIRTUAL_KEY` in `.env` to try this cell.


In [ ]:
if BIFROST_VIRTUAL_KEY:
    bifrost_governed = OpenAI(api_key=BIFROST_VIRTUAL_KEY, base_url=f"{BIFROST_BASE_URL}/v1")

    def bifrost_virtual_key_demo(prompt: str, model: str):
        try:
            raw = bifrost_governed.chat.completions.with_raw_response.create(
                model=model,
                messages=[{"role": "user", "content": prompt}]
            )
            response = raw.parse()
            return {"content": response.choices[0].message.content}
        except Exception as e:
            return {"error": str(e), "type": "budget_exceeded_or_invalid_key"}

    print(bifrost_virtual_key_demo(TEST_PROMPTS["simple"], f"groq/{GROQ_MODEL}"))
else:
    print("BIFROST_VIRTUAL_KEY not set. Create one at http://localhost:8080 -> Virtual Keys to test budget governance.")


### 3.8 Feature 8 — MCP Gateway (DeepWiki + Tavily)

Use case: let the model call external tools — DeepWiki (ask questions about a GitHub repo) and
Tavily (live web search) — through the gateway, instead of wiring tool-calling into every client
app separately. Requires both MCP clients added with **Auto-execute** turned on (Section 2.2).

> **Model choice matters here, and this took real debugging to find:** `openai/gpt-oss-120b`
> and `gpt-oss-20b` (this notebook's Groq models) both **fail with a 400** during Bifrost's
> auto-execute loop — `'reasoning_content' is unsupported`. Bifrost resends the model's own
> reasoning trace back to Groq on the follow-up turn, and Groq's API rejects it for these
> reasoning models. Plain `llama-3.3-70b-versatile` works, but it's the model Groq is retiring
> next month (Section 0.4) — so this section runs on **Mistral** instead, which has no such
> issue and isn't on anyone's deprecation clock.


In [ ]:
def bifrost_mcp_list_tools():
    try:
        response = httpx.get(f"{BIFROST_BASE_URL}/api/mcp/clients", timeout=5)
        return response.json()
    except Exception as e:
        return {"error": str(e)}

def bifrost_mcp_call_with_tools(prompt: str, model: str, servers: list[str]):
    """Code Mode generates TypeScript tool declarations — large token savings at scale."""
    response = httpx.post(
        f"{BIFROST_BASE_URL}/v1/chat/completions",
        json={
            "model":    model,
            "messages": [{"role": "user", "content": prompt}],
            "bifrost": {"mcp": {"enabled": True, "code_mode": True, "servers": servers}}
        },
        timeout=60
    )
    return response.json()

print("=== MCP clients registered in Bifrost ===")
tools = bifrost_mcp_list_tools()
registered = {c["config"]["name"] for c in tools.get("clients", [])}
print("Registered:", registered or "none")

for name, servers in [("DeepWiki", ["deepwiki"]), ("Tavily", ["tavily"])]:
    print()
    print(f"=== MCP call via Bifrost ({name}) ===")
    if servers[0] not in registered:
        print(f"SKIPPED - {servers[0]!r} not registered yet (see Section 2.2).")
        continue
    result = bifrost_mcp_call_with_tools(TEST_PROMPTS[servers[0]], f"mistral/{MISTRAL_MODEL}", servers)
    msg = result.get("choices", [{}])[0].get("message", {})
    if msg.get("tool_calls") and not msg.get("content"):
        print(f"Got a proposed tool call but no final answer - check Auto-execute is ON for {servers[0]}'s tools.")
        print(msg["tool_calls"])
    else:
        print(msg.get("content", result))


> **Not covered here (need paid/enterprise infra, see `bifrost_gateway.ipynb` for the full reference):**
> Guardrails (AWS Bedrock / Azure Content Safety), Cluster Mode (multiple Bifrost nodes),
> Secret Manager integration (HashiCorp Vault / cloud secret managers).


---
<a id="section-4"></a>
## Section 4: A Small RAG Pipeline (Jina Embeddings + Qdrant Cloud + Bifrost Generation)

> Built by hand, not through Bifrost's semantic-cache plugin — so every step (embed, store,
> retrieve, generate) is visible. Generation still goes through Bifrost (Groq/Mistral); only
> embeddings (Jina) and vector search (Qdrant Cloud) sit outside the gateway.


### 4.1 Connect to Qdrant Cloud

Use case: a managed, always-on vector store you don't have to run yourself — just a URL and an
API key from your Qdrant Cloud cluster dashboard (not a local Docker container).


In [ ]:
JINA_API_KEY   = os.getenv("JINA_API_KEY")
QDRANT_URL     = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

assert JINA_API_KEY,   "Missing JINA_API_KEY — set it in .env"
assert QDRANT_URL,     "Missing QDRANT_URL — set it to your Qdrant Cloud cluster URL in .env"
assert QDRANT_API_KEY, "Missing QDRANT_API_KEY — set it to your Qdrant Cloud API key in .env"

from qdrant_client import QdrantClient

# port=443 is required here: qdrant-client defaults to :6333 if no port is given, but Qdrant
# Cloud clusters serve REST over the standard HTTPS port — :6333 gets connection-reset.
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, port=443)
print("Connected. Existing collections:", [c.name for c in qdrant.get_collections().collections])


### 4.2 Jina Embeddings Helper

Use case: turn text into vectors for semantic search. `task="retrieval.passage"` is the adapter
tuned for documents you're storing; `task="retrieval.query"` is tuned for the question you're
searching with — using the right one on each side measurably improves retrieval quality.


In [ ]:
import requests

JINA_MODEL = "jina-embeddings-v4"  # current flagship (2048-dim by default); v3 (1024-dim) also still active

def jina_embed(texts: list[str], task: str = "retrieval.passage") -> list[list[float]]:
    response = requests.post(
        "https://api.jina.ai/v1/embeddings",
        headers={
            "Content-Type":  "application/json",
            "Authorization": f"Bearer {JINA_API_KEY}"
        },
        json={"model": JINA_MODEL, "input": texts, "task": task},
        timeout=30
    )
    response.raise_for_status()
    data = response.json()["data"]
    return [item["embedding"] for item in data]

test_vec = jina_embed(["hello world"], task="retrieval.passage")
JINA_DIM = len(test_vec[0])
print(f"Embedding dimension: {JINA_DIM}")
# Read the dimension from a live call instead of hardcoding it — a hardcoded/wrong dimension
# is a documented Bifrost + Qdrant gotcha (see BIFROST_SETUP.md's lessons-learned table).


### 4.3 Build a Tiny Knowledge Base in Qdrant Cloud

Use case: seed the vector store with a handful of reference documents once, so the retrieval
step in 4.4 has something real to find. Re-running this cell wipes and rebuilds the demo
collection, so the notebook stays repeatable.


In [ ]:
from qdrant_client.models import Distance, VectorParams, PointStruct

COLLECTION = "bifrost_rag_demo"

DOCS = [
    "Bifrost is a Go-based, open-source AI gateway built by Maxim AI, released under Apache 2.0.",
    "Bifrost supports automatic failover across providers, triggered on 5xx errors, 429 rate limits, and 401 auth failures.",
    "Bifrost exposes a single OpenAI-compatible /v1/chat/completions endpoint; the provider is selected via a \"provider/model\" string.",
    "Groq serves LLMs on custom LPU hardware, offering very low-latency inference with a free API tier.",
    "Mistral AI offers La Plateforme with a free tier for models like mistral-large-latest and mistral-small-latest.",
    "Qdrant Cloud is a managed vector database used here for RAG retrieval, reached over a URL + API key.",
    "Jina AI provides an embeddings API (jina-embeddings-v4) with task-specific adapters for retrieval and classification."
]

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)
qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=JINA_DIM, distance=Distance.COSINE)
)

vectors = jina_embed(DOCS, task="retrieval.passage")
qdrant.upsert(
    collection_name=COLLECTION,
    points=[
        PointStruct(id=i, vector=vec, payload={"text": doc})
        for i, (vec, doc) in enumerate(zip(vectors, DOCS))
    ]
)
print(f"Indexed {len(DOCS)} documents into Qdrant Cloud collection \"{COLLECTION}\"")


### 4.4 Retrieve + Generate (Full RAG Loop, Generation via Bifrost)

Use case: the actual RAG pattern — embed the question, retrieve the closest stored facts from
Qdrant, then ask an LLM (through Bifrost, so you still get failover/observability/etc. for
free) to answer using only that retrieved context.


In [ ]:
def rag_query(question: str, model: str = f"groq/{GROQ_MODEL}", top_k: int = 3) -> dict:
    query_vec = jina_embed([question], task="retrieval.query")[0]

    hits = qdrant.query_points(
        collection_name=COLLECTION,
        query=query_vec,
        limit=top_k
    ).points

    context = "\n".join(f"- {hit.payload['text']}" for hit in hits)
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer isn't in the context, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )

    answer = bifrost_call(prompt, model)

    return {
        "question":  question,
        "retrieved": [hit.payload["text"] for hit in hits],
        "scores":    [round(hit.score, 4) for hit in hits],
        "answer":    answer["content"],
        "provider":  answer["provider"]
    }

result = rag_query("What hardware does Groq use for inference?")
print("Question :", result["question"])
print("Retrieved:")
for text, score in zip(result["retrieved"], result["scores"]):
    print(f"  [{score}] {text}")
print()
print("Answer   :", result["answer"])
print("Provider :", result["provider"])


---
## Summary

| Covered | Skipped (needs paid/enterprise infra) |
|---|---|
| Vanilla Groq/Mistral SDK baseline | Guardrails (AWS Bedrock / Azure) |
| Drop-in unified Bifrost client (Web UI configured) | Cluster mode (multi-node) |
| Automatic failover, weighted load balancing | Secret Manager (Vault / cloud) |
| Unified streaming | Prometheus `/metrics` (needs plugin binary) |
| Logs API, optional OpenTelemetry tracing | |
| Virtual keys & budgets | |
| MCP gateway (DeepWiki + Tavily) | |
| Hand-built RAG: Jina embeddings + Qdrant Cloud + Bifrost generation | |

For the enterprise-tier features, see `bifrost_gateway.ipynb`.
